# Spectral analysis: lab notebook

**Statistical Modelling and Forecasting, Multivariate and advanced methods**

Run the cells from top to bottom (Runtime > Run all). NumPy, pandas, SciPy, statsmodels and scikit-learn are already installed in Colab. Read each section, run it, and compare your output with the expected output at the end of the notebook.

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # keeps the output tidy

In [ ]:
import numpy as np
from scipy import signal

rng = np.random.default_rng(5)
n = 240
t = np.arange(1, n + 1)

def periodogram(y):
    z = y - y.mean()
    F = np.fft.rfft(z)
    freq = np.fft.rfftfreq(len(y))
    return freq[1:], (np.abs(F) ** 2 / (2 * np.pi * len(y)))[1:]

### 1. A noisy sinusoid with frequency 1/12 (a monthly seasonal cycle)

In [ ]:
# 1. A noisy sinusoid with frequency 1/12 (a monthly seasonal cycle)
y = 3 * np.sin(2 * np.pi * t / 12) + 1.5 * rng.standard_normal(n)
f, I = periodogram(y)
j = np.argmax(I)
print("peak at frequency", round(f[j], 4), "-> period", round(1 / f[j], 2), "| Fourier index j =", j + 1, "= n/12 =", n // 12)

### 2. The periodogram integrates to the variance

In [ ]:
# 2. The periodogram integrates to the variance
print("2 * sum(I) * 2*pi/n =", round(2 * I.sum() * 2 * np.pi / n, 3), "| sample variance =", round(y.var(), 3))

### 3. Hand formula against the FFT modulus

In [ ]:
# 3. Hand formula against the FFT modulus
print("hand I(w_j) * 2*pi*n =", round(I[j] * 2 * np.pi * n, 1), "| |FFT|^2 =", round(np.abs(np.fft.rfft(y - y.mean())[j + 1]) ** 2, 1))

### 4. The raw periodogram is not consistent: smooth it with a Daniell kernel

In [ ]:
# 4. The raw periodogram is not consistent: smooth it with a Daniell kernel
def daniell(a, m):
    k = np.ones(2 * m + 1) / (2 * m + 1)
    return np.convolve(a, k, mode="same")
w = rng.standard_normal(n)
fw, Iw = periodogram(w)
print("\nwhite noise: true spectrum 1/(2*pi) =", round(1 / (2 * np.pi), 4))
for m in (0, 2, 6):
    S = daniell(Iw, m) if m else Iw
    print(f"  m={m}: mean {S.mean():.4f}, sd {S.std():.4f}")

### 5. AR(1): the shape of the spectrum follows phi

In [ ]:
# 5. AR(1): the shape of the spectrum follows phi
for phi in (0.8, -0.8):
    a = np.zeros(n + 100)
    e = rng.standard_normal(n + 100)
    for i in range(1, n + 100):
        a[i] = phi * a[i - 1] + e[i]
    fa, Ia = periodogram(a[100:])
    S = daniell(Ia, 6)
    theo = 1 / (2 * np.pi * (1 - 2 * phi * np.cos(2 * np.pi * fa) + phi ** 2))
    corr = np.corrcoef(np.log(S[3:-3]), np.log(theo[3:-3]))[0, 1]
    print(f"AR(1) phi={phi:+.1f}: power at lowest vs highest frequency =", round(S[:10].mean() / S[-10:].mean(), 2), "| log-correlation with theory:", round(corr, 3))

### 6. Two close frequencies need a long record to separate

In [ ]:
# 6. Two close frequencies need a long record to separate
for N in (60, 240):
    tt = np.arange(1, N + 1)
    s = np.sin(2 * np.pi * 0.10 * tt) + np.sin(2 * np.pi * 0.12 * tt) + 0.3 * rng.standard_normal(N)
    ff, II = periodogram(s)
    peaks, _ = signal.find_peaks(II, height=0.3 * II.max())
    print(f"n={N}: resolution 1/n={1 / N:.4f}; peaks found at", np.round(ff[peaks], 3))

## Expected output

Running all cells above should print:

```
peak at frequency 0.0833 -> period 12.0 | Fourier index j = 20 = n/12 = 20
2 * sum(I) * 2*pi/n = 6.81 | sample variance = 6.807
hand I(w_j) * 2*pi*n = 133280.6 | |FFT|^2 = 133280.6

white noise: true spectrum 1/(2*pi) = 0.1592
  m=0: mean 0.1385, sd 0.1460
  m=2: mean 0.1368, sd 0.0649
  m=6: mean 0.1351, sd 0.0392
AR(1) phi=+0.8: power at lowest vs highest frequency = 93.21 | log-correlation with theory: 0.97
AR(1) phi=-0.8: power at lowest vs highest frequency = 0.01 | log-correlation with theory: 0.972
n=60: resolution 1/n=0.0167; peaks found at [0.1]
n=240: resolution 1/n=0.0042; peaks found at [0.1   0.121]
```

## Practice exercises

Try these in the cells below.

1. Choose "Seasonal cycle, period 12" and find the peak. Convert the peak frequency to a period.
2. Choose AR(1) and move φ from +0.9 to −0.9. Where does the power sit?
3. Increase the smoothing bandwidth for the noisy sinusoid. When do two close peaks merge?

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3
